# Gate 2 — A1 (MBHT + B1) SMOKE TEST trên RetailRocket

## ⚠ ĐỌC TRƯỚC: notebook này KHÔNG quyết định B1 sống hay chết

**Tiêu chí PASS:**
1. Không NaN / Inf trong training.
2. Số param A1 tăng **đúng 1,984** (= |T| 31 × hidden 64).
3. **Đo và ghi lại sàn nhiễu giữa các run** (không còn là bit-exact — xem bên dưới).

**Go/no-go cho B1 KHÔNG nằm trong notebook này.** Nó là: hiệu **A1 − A0 phải vượt độ lệch
giữa các run**, đo bằng **multi-seed mean±std**, trên **Tmall**. Một run đơn lẻ không kết luận
được gì.

---

### Vì sao tiêu chí (3) không còn là bit-exact

Đã thử và **thất bại có bằng chứng**: hai run A0 cùng seed 2020, cùng code, cùng dữ liệu vẫn
cho kết quả khác nhau, thậm chí dừng ở số epoch khác nhau (26 vs 25).

| metric | A0 run1 | A0 run2 | Δ |
|---|---:|---:|---:|
| recall@5 | 0.9440 | 0.9375 | **−0.0065** |
| ndcg@5 | 0.9299 | 0.9272 | −0.0027 |
| recall@10 | 0.9495 | 0.9480 | −0.0015 |
| **ndcg@10** (metric chính) | 0.9316 | 0.9306 | **−0.0010** |
| mrr@10 | 0.9259 | 0.9252 | −0.0007 |

Đã loại trừ tuần tự: `init_seed` có seed đủ và bật `cudnn.deterministic=True`; `fresh_split()`
tất định (kiểm chứng riêng); B1 tắt ở cả hai run.

**Nguồn nhiễu nằm trong `build_Gs_unique()` của code gốc — phần hypergraph bị cấm sửa:**

1. `H[idx, unique_idx] = metrics[idx]` (`mbht.py:503`) — `unique_idx` **có thể trùng lặp** khi
   một item xuất hiện ở nhiều vị trí. Gán theo chỉ số trùng lặp trên CUDA là **không tất định**
   theo đúng tài liệu PyTorch.
2. `item_sim[item_sim < 0] = 0.01` rồi `torch.topk(...)` — ép mọi similarity âm về đúng `0.01`,
   tạo hàng loạt giá trị hoà nhau; `topk` chọn giữa các giá trị hoà trên CUDA cũng không tất định.
   Với dữ liệu thưa 99.996% như RetailRocket, đây nhiều khả năng là nguồn chính.

Ép tất định (`torch.use_deterministic_algorithms(True)`) sẽ báo lỗi ngay tại (1) và buộc phải sửa
cấu trúc hypergraph — **vi phạm ràng buộc "hypergraph UNCHANGED"** và làm A0 khác model published.
Nên ta chấp nhận nhiễu và chuyển sang tiêu chí thống kê.

---

### ⚠ Sàn nhiễu này áp cho MỌI so sánh về sau

| metric | sàn nhiễu quan sát (n=2) |
|---|---:|
| ndcg@10 | **±0.0010** |
| recall@5 | **±0.0065** (≈0.69% tương đối) |

**Kết luận rút từ 1–2 run là KHÔNG đáng tin.**

Ba mẫu A0 hiện có — Gate 1 `0.9224` (trước fix gating_bias), run1 `0.9316`, run2 `0.9306` —
**không** đủ để kết luận fix gating_bias làm tăng điểm. n quá nhỏ, chưa tách được khỏi nhiễu.

---

### Vì sao RetailRocket không dùng để quyết định

Test set của RetailRocket **không chứa một sự kiện mua nào**. Train có 20 loại transition, test
chỉ có 6; `CART→BUY` chiếm 6.68% train nhưng **0% test**. Tín hiệu B1 nhắm tới gần như không tồn
tại lúc inference trên bộ này. Tmall/IJCAI có đủ (6.85% / 7.82%).

Commit ghim: `3543355`

## 1. GPU + mount Drive

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

## 2. Đường dẫn (giống Gate 1 — dùng lại data/checkpoint đã có trên Drive)

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/DeAnThS'
DATA_DIR     = os.path.join(PROJECT_ROOT, 'dataset')
CKPT_DIR     = os.path.join(PROJECT_ROOT, 'checkpoints')
LOG_DIR      = os.path.join(PROJECT_ROOT, 'logs')
RUN_META_DIR = os.path.join(PROJECT_ROOT, 'run_meta')
CODE_DIR     = '/content/MBHT-KDD22'

for d in [DATA_DIR, CKPT_DIR, LOG_DIR, RUN_META_DIR]:
    os.makedirs(d, exist_ok=True)

DATASET_ROOT = os.path.join(DATA_DIR, 'MBHT_dataset')
assert os.path.isdir(DATASET_ROOT), f'Chua co {DATASET_ROOT} -- chay notebook Gate 1-2 truoc.'

REPO_URL      = 'https://github.com/nguyenlmhcm/MBHT-KDD22.git'
BRANCH        = 'bt-mbht'
PINNED_COMMIT = '3543355'   # B1 + gating_bias init fix
DATASET_NAME  = 'retail_beh'

print('DATASET_ROOT:', DATASET_ROOT)
print('PINNED_COMMIT:', PINNED_COMMIT)

## 3. Clone code đã có B1 + cài dependency

In [ ]:
import subprocess, time

# The dependency cell below does `%cd {CODE_DIR}`, so re-running this cell
# would delete the directory the process is standing in -- after which every
# git call fails with "Unable to read current working directory". Step out
# first so a re-run is always safe.
os.chdir('/content')

if os.path.isdir(CODE_DIR):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)

for attempt in range(1, 6):
    r = subprocess.run(['git','clone','-b',BRANCH,REPO_URL,CODE_DIR], capture_output=True, text=True)
    if r.returncode == 0:
        print(f'Clone OK (attempt {attempt})'); break
    print(f'attempt {attempt} failed:', r.stderr.strip())
    if os.path.isdir(CODE_DIR):
        subprocess.run(['rm','-rf',CODE_DIR], check=True)
    if attempt == 5:
        raise RuntimeError('git clone failed 5x')
    time.sleep(10*attempt)

subprocess.run(['git','-C',CODE_DIR,'checkout',PINNED_COMMIT], check=True)
head = subprocess.run(['git','-C',CODE_DIR,'rev-parse','HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert head.startswith(PINNED_COMMIT), f'{head} != {PINNED_COMMIT}'
print('commit:', head)

In [ ]:
%cd {CODE_DIR}
!pip install -q hyperopt pandas tqdm scikit_learn pyyaml colorlog colorama tensorboard

## 4. Chạy unit test B1 trên chính máy Colab

29 test này đã pass trên VPS; chạy lại ở đây để chắc chắn không có gì vỡ do khác phiên bản thư viện.

In [ ]:
!python tests/test_transition_utils.py
print()
!python tests/test_b1_integration.py

## 5. Load dataset + in mapping behavior (xác nhận n_types = 5, |T| = 31)

In [ ]:
from recbole.config import Config
from recbole.data import create_dataset
from recbole.model.transition_utils import transition_vocab_size

base_cfg = {
    'data_path': DATASET_ROOT,
    'USER_ID_FIELD': 'session_id',
    'load_col': None,
    'neg_sampling': None,
    'benchmark_filename': ['train', 'test'],
    'alias_of_item_id': ['item_id_list'],
    'MAX_ITEM_LIST_LENGTH': 200,
}
schema_config = Config(model='MBHT', dataset=DATASET_NAME, config_dict=base_cfg)
dataset = create_dataset(schema_config)

type_map = dataset.field2token_id['item_type_list']
n_types  = len(type_map)
print('field2token_id[item_type_list] =', type_map)
print('n_types =', n_types)
print('|T|     =', transition_vocab_size(n_types))
assert n_types == 5, f'RetailRocket phai co 5 type, dang thay {n_types}'
assert transition_vocab_size(n_types) == 31

def fresh_split(cfg=None):
    """A pristine dataset + split for ONE consumer. Never share the result.

    Two independent reasons it cannot be reused:

    * dataset.build() is not idempotent -- it swaps inter_feat from a
      DataFrame to an Interaction in place, so a second call reads `.values`
      off a tensor and dies with "data must be a sequence (got
      builtin_function_or_method)".
    * TrainDataLoader._shuffle() calls dataset.shuffle(), which reorders the
      split IN PLACE once per epoch. A split handed to a second run would
      therefore start from whatever order the first run left behind, and the
      two runs could not be bit-identical.

    Rebuilding costs a few seconds and buys freedom from both.
    """
    ds = create_dataset(cfg if cfg is not None else schema_config)
    tr, te = ds.build()
    return ds, tr, te


_probe_ds, _probe_tr, _probe_te = fresh_split()
print('train rows:', len(_probe_tr), ' test rows:', len(_probe_te))
del _probe_ds, _probe_tr, _probe_te

## 6. Bảng tần suất transition (checklist 15.1)

Chạy trên chính train dataloader, qua đúng hàm model dùng.

In [ ]:
import collections, torch
from recbole.data.utils import get_dataloader, create_samplers
from recbole.model.transition_utils import (
    build_transition_seq, count_transitions, format_transition_table,
)

id2token = dataset.field2id_token['item_type_list']
counter = collections.Counter()

tmp_cfg = Config(model='MBHT', dataset=DATASET_NAME,
                 config_dict={**base_cfg, 'train_batch_size': 64, 'eval_batch_size': 128})
tmp_ds, tmp_tr, tmp_te = fresh_split(tmp_cfg)
tr_sampler, te_sampler = create_samplers(tmp_cfg, tmp_ds, [tmp_tr, tmp_te])
# shuffle=False: counting must not reorder anything
tmp_loader = get_dataloader(tmp_cfg, 'train')(tmp_cfg, tmp_tr, tr_sampler, shuffle=False)

for batch in tmp_loader:
    item_seq = batch['item_id_list']
    type_seq = batch['item_type_list']
    counter = count_transitions(
        build_transition_seq(type_seq, item_seq > 0, n_types), counter=counter
    )

print(format_transition_table(counter, n_types, id2token))

## 7. Hàm chạy một run (dùng chung cho A0-rerun và A1)

Mọi thứ giống hệt Gate 1 trừ đúng một biến: `enable_transition_embedding`.

In [ ]:
import glob, hashlib, json
from logging import getLogger
from recbole.model.sequential_recommender.mbht import MBHT
from recbole.utils import init_logger, init_seed, get_trainer, set_color

FROZEN = {
    'USER_ID_FIELD': 'session_id',
    'load_col': None,
    'neg_sampling': None,
    'benchmark_filename': ['train', 'test'],
    'alias_of_item_id': ['item_id_list'],
    'topk': [5, 10, 101],
    'metrics': ['Recall', 'NDCG', 'MRR'],
    'valid_metric': 'NDCG@10',
    'eval_args': {'mode': 'full', 'order': 'TO'},
    'MAX_ITEM_LIST_LENGTH': 200,
    'train_batch_size': 64,
    'eval_batch_size': 128,
    'hyper_len': 6,
    'scales': [5, 4, 20],
    'enable_hg': 1,
    'enable_ms': 1,
    'customized_eval': 1,
    'abaltion': '',
}

def run(tag, enable_b1, seed=2020):
    ckpt = os.path.join(CKPT_DIR, tag); os.makedirs(ckpt, exist_ok=True)
    logd = os.path.join(LOG_DIR, tag);  os.makedirs(logd, exist_ok=True)

    cfg_dict = {**FROZEN,
                'data_path': DATASET_ROOT,
                'checkpoint_dir': ckpt,
                'gpu_id': 0,
                'seed': seed,
                'enable_transition_embedding': int(enable_b1)}
    config = Config(model='MBHT', dataset=DATASET_NAME, config_dict=cfg_dict)
    init_seed(config['seed'], config['reproducibility'])
    init_logger(config, log_root=logd)
    logger = getLogger()
    logger.info(f'TAG={tag} commit={PINNED_COMMIT} enable_transition_embedding={int(enable_b1)}')

    # Own split: the train dataloader shuffles it in place every epoch, so a
    # shared one would leak this run's final ordering into the next.
    run_ds, tr, te = fresh_split(config)
    s_tr, s_te = create_samplers(config, run_ds, [tr, te])
    train_data = get_dataloader(config, 'train')(config, tr, s_tr, shuffle=True)
    test_data  = get_dataloader(config, 'test')(config, te, s_te, shuffle=False)

    model = MBHT(config, train_data.dataset).to(config['device'])
    n_params = sum(p.numel() for p in model.parameters())
    print(f'[{tag}] enable_transition_embedding={model.enable_transition_embedding} '
          f'n_behavior_types={model.n_behavior_types} |T|={model.n_transitions} '
          f'params={n_params}')

    trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)
    prev = sorted(glob.glob(os.path.join(ckpt, '*.pth')), key=os.path.getmtime)
    if prev:
        print(f'[{tag}] resuming from {prev[-1]}')
        trainer.resume_checkpoint(prev[-1])

    score, result = trainer.fit(train_data, test_data, saved=True, show_progress=config['show_progress'])
    if result is None:
        raise RuntimeError(
            f'[{tag}] trainer returned no result. This happens when resuming a checkpoint '
            f'that already finished training: fit() skips the epoch loop and never sets '
            f'best_valid_result. Delete {ckpt} and run again.')
    print(set_color(f'[{tag}] test result', 'yellow') + f': {result}')

    # Metrics are rounded to metric_decimal_place (4), so comparing them can
    # only detect differences above 1e-4. Hash the trained weights too, which
    # is what "bit-exact" actually means.
    param_sig = hashlib.sha256()
    for name, p in sorted(model.state_dict().items()):
        param_sig.update(name.encode())
        param_sig.update(p.detach().cpu().numpy().tobytes())
    param_hash = param_sig.hexdigest()[:16]
    print(f'[{tag}] param hash: {param_hash}')

    with open(os.path.join(RUN_META_DIR, f'{tag}.json'), 'w') as f:
        json.dump({'tag': tag, 'commit': PINNED_COMMIT, 'seed': seed,
                   'enable_transition_embedding': int(enable_b1),
                   'n_params': n_params, 'param_hash': param_hash,
                   'result': {k: float(v) for k, v in result.items()}},
                  f, indent=2)
    return n_params, result, param_hash

## 8. PASS #2 — param delta đúng 1,984

Dựng cả hai model (chưa train) và so số param.

In [ ]:
cfg_off = Config(model='MBHT', dataset=DATASET_NAME,
                 config_dict={**FROZEN, 'data_path': DATASET_ROOT, 'seed': 2020,
                              'enable_transition_embedding': 0})
cfg_on  = Config(model='MBHT', dataset=DATASET_NAME,
                 config_dict={**FROZEN, 'data_path': DATASET_ROOT, 'seed': 2020,
                              'enable_transition_embedding': 1})

probe_ds, tr, te = fresh_split(cfg_off)
s_tr, s_te = create_samplers(cfg_off, probe_ds, [tr, te])
probe = get_dataloader(cfg_off, 'train')(cfg_off, tr, s_tr, shuffle=False)

init_seed(2020, True); m_off = MBHT(cfg_off, probe.dataset)
init_seed(2020, True); m_on  = MBHT(cfg_on,  probe.dataset)

p_off = sum(p.numel() for p in m_off.parameters())
p_on  = sum(p.numel() for p in m_on.parameters())
delta = p_on - p_off
print(f'A0 (flag=0) params : {p_off}')
print(f'A1 (flag=1) params : {p_on}')
print(f'delta              : {delta}   (ky vong 31*64 = {31*64})')
assert delta == 31*64, f'PASS #2 FAIL: delta={delta}'

new_keys = set(m_on.state_dict()) - set(m_off.state_dict())
print('key moi:', new_keys)
assert new_keys == {'transition_embedding.weight'}

# Moi tham so A0 phai y nguyen khi bat flag -- KHONG loai tru gi.
# gating_bias tung phai loai tru vi no la bo nho chua khoi tao; sau fix
# 3543355 no da xac dinh theo seed, nen gio kiem tra duoc day du.
sd_off, sd_on = m_off.state_dict(), m_on.state_dict()
diff = [k for k in sd_off if not torch.equal(sd_off[k], sd_on[k])]
print('tham so A0 bi lech khi bat flag:', diff)
assert not diff, f'bat B1 lam lech {diff} -- ablation bi nhieu'
print('\nPASS #2 OK')

## 9. PASS #3 — đo sàn nhiễu giữa các run (KHÔNG còn assert bit-exact)

Chạy A0 **hai lần** cùng seed để **đo** độ lệch giữa các run, chứ không phải để đòi chúng bằng
nhau. Bit-exact là bất khả thi mà không sửa hypergraph (xem phần đầu notebook).

Con số đo được ở đây là **ngưỡng** mà mọi hiệu A1−A0 về sau phải vượt qua mới đáng tin.

In [ ]:
GATE1_A0 = {'recall@5':0.9305,'recall@10':0.9375,'ndcg@5':0.9202,
            'ndcg@10':0.9224,'mrr@5':0.9167,'mrr@10':0.9177}   # tham khao, KHONG phai tieu chi

p_a0, res_a0, h_a0 = run('A0rerun1-MBHT-retail_beh-seed2020', enable_b1=False, seed=2020)
p_a0b, res_a0b, h_a0b = run('A0rerun2-MBHT-retail_beh-seed2020', enable_b1=False, seed=2020)

# PASS #3: DO san nhieu, khong doi hai run bang nhau.
NOISE = {k: abs(float(res_a0b[k]) - float(res_a0[k])) for k in res_a0}

print('\n=== PASS #3: san nhieu giua hai run A0 cung seed ===')
print(f'{"metric":>12} {"run1":>10} {"run2":>10} {"|delta|":>12}')
for k in sorted(res_a0):
    print(f'{k:>12} {float(res_a0[k]):>10.6f} {float(res_a0b[k]):>10.6f} {NOISE[k]:>12.4f}')

print(f'\nparam hash run1: {h_a0}')
print(f'param hash run2: {h_a0b}')
print(f'bit-exact? {h_a0 == h_a0b}  <-- ky vong False; nguon nhieu nam trong hypergraph goc')

# Chi assert nhung gi thuc su phai dung
for name, r in (('run1', res_a0), ('run2', res_a0b)):
    bad = [k for k, v in r.items() if v != v or abs(float(v)) == float('inf')]
    assert not bad, f'PASS #1 FAIL: {name} co NaN/Inf o {bad}'
print('\nPASS #1 OK -- khong NaN/Inf')
print('PASS #3 OK -- da do va ghi lai san nhieu (khong phai bit-exact)')

with open(os.path.join(RUN_META_DIR, 'noise_floor_retail_beh.json'), 'w') as f:
    json.dump({'dataset': DATASET_NAME, 'seed': 2020, 'n_runs': 2,
               'source': 'build_Gs_unique: duplicate-index assign (mbht.py:503) + tied topk on CUDA',
               'run1': {k: float(v) for k, v in res_a0.items()},
               'run2': {k: float(v) for k, v in res_a0b.items()},
               'abs_delta': NOISE}, f, indent=2)
print(f"\nDa luu: {os.path.join(RUN_META_DIR, 'noise_floor_retail_beh.json')}")

print('\n--- Tham khao: A0 (sau fix gating_bias) vs so Gate 1 (truoc fix) ---')
print('KHONG ket luan gi tu bang nay: n=1 truoc fix, n=2 sau fix, deu nho hon san nhieu can thiet.')
print(f'{"metric":>12} {"Gate1":>9} {"run1":>9} {"run2":>9}')
for k in GATE1_A0:
    print(f'{k:>12} {GATE1_A0[k]:>9.4f} {float(res_a0[k]):>9.4f} {float(res_a0b[k]):>9.4f}')

## 10. A1 — bật B1 (chỉ log, KHÔNG kết luận go/no-go)

In [ ]:
p_a1, res_a1, h_a1 = run('A1-MBHT-B1-retail_beh-seed2020', enable_b1=True, seed=2020)

bad = [k for k, v in res_a1.items() if v != v or abs(float(v)) == float('inf')]
assert not bad, f'PASS #1 FAIL: A1 co NaN/Inf o {bad}'
print('\nPASS #1 OK -- A1 khong NaN/Inf')
print(f'PASS #2: params A0={p_a0}  A1={p_a1}  delta={p_a1 - p_a0}  (ky vong {31*64})')

# A0 lay trung binh hai run de bot nhieu mot chut
print('\n=== A1 vs A0 (mean cua 2 run) -- doi chieu voi SAN NHIEU ===')
print(f'{"metric":>12} {"A0 mean":>9} {"A1":>9} {"delta":>10} {"noise":>9}   ket luan')
for k in sorted(res_a0):
    a0m = (float(res_a0[k]) + float(res_a0b[k])) / 2
    a1v = float(res_a1[k])
    d = a1v - a0m
    n = NOISE[k]
    verdict = 'TRONG NHIEU (khong ket luan duoc)' if abs(d) <= n else 'vuot san nhieu -- van can multi-seed'
    print(f'{k:>12} {a0m:>9.4f} {a1v:>9.4f} {d:>+10.4f} {n:>9.4f}   {verdict}')

print('\n' + '='*70)
print('KHONG KET LUAN B1 TU BANG TREN. Hai ly do doc lap:')
print('  1. RetailRocket test set khong co su kien mua nao (CART->BUY = 0%),')
print('     nen tin hieu B1 nham toi gan nhu khong ton tai luc inference.')
print('  2. Day la 1 run cua A1 va 2 run cua A0 -- duoi muc can thiet de')
print('     tach hieu ung khoi nhieu.')
print('Go/no-go that su: multi-seed mean+-std tren Tmall.')
print('='*70)

## 11. Cần dán về những gì

1. Output test ở cell 4 (`20/20` và `10/10`).
2. `field2token_id`, n_types, |T| (cell 5).
3. Bảng tần suất transition (cell 6).
4. **PASS #2** — param delta (cell 8).
5. **PASS #3** — bảng sàn nhiễu giữa hai run A0 (cell 9).
6. Bảng A1 vs A0 kèm cột "ket luan" (cell 10).
7. Bất kỳ NaN/traceback nào — **PASS #1**.

Sau đó: chốt số seed cho Tmall rồi mới chạy loạt multi-seed.